In [5]:
import os
import sys
from dotenv import load_dotenv
from utils.helper_functions import *
from utils.evaluate_rag import *

load_dotenv()

path="data/hyde_rag.pdf"

In [6]:
class HyDERetriever:
    def __init__(self,file_path,chunk_size=500,chunk_overlap=100):
        self.llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.6,max_completion_tokens=5000)
        self.embeddings=OpenAIEmbeddings(model='text-embedding-3-small')
        self.chunk_size=chunk_size,
        self.chunk_overlap=chunk_overlap,
        self.vectorstore=encode_pdf(file_path,self.chunk_size,self.chunk_overlap)

        self.HyDEPrompts=PromptTemplate(
            input_variables=["query","chunk_size"],
            template="""
                        You are an expert writer.
                        Given the following question, write a concise, factual document that would likely answer it.
                        The document should resemble a passage from a textbook, article, or knowledge base.
                        Do not mention that this is a hypothetical document.
                        Do not include phrases like "I think" or "As an AI".
                        the document size has be exactly {chunk_size} characters.

                    Question:
                    {query}
                    Hypothetical Document:
                    """
                    )
        self.HyDEChain=self.HyDEChain|self.HyDEPrompts

    def generate_hypothetical_documents(self,query:str):
        input_variable={"query":query,"chunk_size":self.chunk_size}
        return self.HyDEChain.invoke(input_variable).content

    def retrieve(self,query:str,k=3):
        hypothetical_docs=self.generate_hypothetical_documents(self,query)
        similar_docs=self.vectorstore.similarity_search(self.hypothetical_documents,k)
        return similar_docs,hypothetical_docs

        